# Quantum Computing for Image Compression
## A Tutorial Using PennyLane and polyfit-image-compress

This notebook demonstrates how the classical least-squares solver at the heart of
polynomial surface fitting can be reformulated as a quantum optimization problem
using the Variational Quantum Linear Solver (VQLS).

**Important**: The quantum module is educational — simulators are SLOWER than classical
NumPy operations. The value is in understanding how quantum algorithms work, not performance.

In [ ]:
import numpy as np
from skimage import data
import matplotlib.pyplot as plt

from polyfit_compress import LeastSquaresCompressor, LinearModel
from polyfit_compress.metrics import psnr
from polyfit_compress.quantum import VQLSSolver, VQLSConfig, HybridCompressor, QuantumFeatureMap

In [ ]:
import numpy as np
from skimage import data
import matplotlib.pyplot as plt

from polyfit_compress import LeastSquaresCompressor, LinearModel, QuadraticModel
from polyfit_compress.metrics import psnr, mse
from polyfit_compress.quantum import VQLSSolver, VQLSConfig, HybridCompressor, QuantumFeatureMap

In [ ]:
# Load test image and compress classically
img = data.camera()[::4, ::4]  # Downsample to 128x128 for speed
compressor = LeastSquaresCompressor(model=LinearModel(), block_size=4)
result_classical = compressor.compress(img)
print(f"Classical PSNR: {psnr(img, result_classical.reconstructed):.2f} dB")
print(f"Classical Ratio: {result_classical.compression_ratio:.2f}x")

## What is VQLS?

The Variational Quantum Linear Solver solves **Ax = b** using a quantum circuit.

Instead of computing `x = A⁺b` directly (classical approach), VQLS:
1. Prepares a parameterized quantum state |ψ(θ)⟩
2. Minimizes a cost function that measures how well A|ψ(θ)⟩ approximates |b⟩
3. Extracts the solution from the optimized quantum state

The circuit uses StronglyEntanglingLayers as the ansatz — a parameterized template
that can approximate arbitrary quantum states.

In [ ]:
# Solve a single block with VQLS
model = LinearModel()
A = model.build_design_matrix(4)  # 4x4 block -> 16x3 design matrix

# Take a single block from the image
block = img[:4, :4].flatten().astype(np.float64)

# Classical solve
classical_coeffs = np.linalg.lstsq(A, block, rcond=None)[0]

# Quantum solve
config = VQLSConfig(n_layers=4, max_iterations=100, stepsize=0.1)
solver = VQLSSolver(config)
quantum_coeffs, converged = solver.solve(A, block)

print(f"Classical coefficients: {classical_coeffs}")
print(f"Quantum coefficients:  {quantum_coeffs}")
print(f"Converged: {converged}")

# Compare direction (cosine similarity)
cos_sim = np.dot(classical_coeffs, quantum_coeffs) / (
    np.linalg.norm(classical_coeffs) * np.linalg.norm(quantum_coeffs)
)
print(f"Cosine similarity: {cos_sim:.4f}")

## Hybrid Classical-Quantum Pipeline

The `HybridCompressor` wraps the classical compressor and replaces the
pseudoinverse solve step with VQLS — block by block. If VQLS doesn't
converge for a block, it automatically falls back to the classical solver.

In [ ]:
# Compress with hybrid pipeline
config = VQLSConfig(n_layers=4, max_iterations=50, stepsize=0.1)
small_img = img[:16, :16]  # Use very small image for demo speed

converged_blocks = []
def progress(block_idx, total, converged):
    converged_blocks.append(converged)
    if block_idx % 4 == 0:
        print(f"Block {block_idx}/{total} — {'VQLS' if converged else 'Classical fallback'}")

hybrid = HybridCompressor(
    model=LinearModel(),
    block_size=4,
    vqls_config=config,
    progress_callback=progress,
)
result_hybrid = hybrid.compress(small_img)

n_converged = sum(converged_blocks)
print(f"\nConverged: {n_converged}/{len(converged_blocks)} blocks")
print(f"Hybrid PSNR: {psnr(small_img, result_hybrid.reconstructed):.2f} dB")

## Quantum Feature Maps

Instead of polynomial basis functions (1, x, y, xy, x², y²), we can use
quantum circuits to create features. Each pixel coordinate (x, y) is encoded
as rotation angles, and expectation values of Pauli operators form the design matrix.

In [ ]:
# Compare classical vs quantum design matrices
classical_A = LinearModel().build_design_matrix(4)
quantum_A = QuantumFeatureMap(n_features=3).build_design_matrix(4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(classical_A, aspect='auto', cmap='viridis')
ax1.set_title("Classical Design Matrix (Linear)")
ax1.set_xlabel("Coefficients")
ax1.set_ylabel("Pixels")

ax2.imshow(quantum_A, aspect='auto', cmap='viridis')
ax2.set_title("Quantum Design Matrix")
ax2.set_xlabel("Features")
ax2.set_ylabel("Pixels")

plt.tight_layout()
plt.show()

## Honest Performance Comparison

Let's be explicit: quantum simulation is SLOWER than classical NumPy.
The value of this module is educational, not practical.

In [ ]:
import time

small_img = img[:16, :16]
configs = [
    ("Classical (pinv)", None),
    ("VQLS (50 iter)", VQLSConfig(max_iterations=50)),
    ("VQLS (100 iter)", VQLSConfig(max_iterations=100)),
]

print(f"{'Method':<25} {'Time (s)':<12} {'PSNR (dB)':<12}")
print("-" * 50)

for name, vqls_config in configs:
    if vqls_config is None:
        comp = LeastSquaresCompressor(model=LinearModel(), block_size=4)
    else:
        comp = HybridCompressor(model=LinearModel(), block_size=4, vqls_config=vqls_config)

    t0 = time.perf_counter()
    result = comp.compress(small_img)
    elapsed = time.perf_counter() - t0

    quality = psnr(small_img, result.reconstructed)
    print(f"{name:<25} {elapsed:<12.3f} {quality:<12.2f}")

## Key Takeaways

1. **VQLS works** — it finds coefficients in the same direction as classical least squares
2. **Simulators are slower** — 100-1000x overhead vs NumPy (expected on classical hardware)
3. **Convergence is not guaranteed** — the fallback mechanism ensures valid output always
4. **Educational value** — understanding how Ax=b maps to quantum circuits is the real goal
5. **Future potential** — with error-corrected quantum hardware, VQLS could offer speedup for large systems